# LLM-as-Judge 评估

前置知识：C5 系统评估与优化、本章 Notebook 2

本节目标：掌握用 LLM 做评估的核心技术——手写 Judge Prompt、对比 CoT 打分与直接打分、校准 Judge 与人工标注的一致性。

## 一、LLM-as-Judge 原理

### 核心思想

LLM-as-Judge 的核心思想是**用大语言模型替代昂贵的人工标注**，对 RAG 系统的输出进行自动化评估。与传统的 BLEU、ROUGE 等基于文本重叠的指标不同，LLM-as-Judge 能够理解语义，从而更接近人类的判断。

### 与 C5 的区别

在 C5「系统评估与优化」中，我们介绍了 LLM-as-Judge 的基本概念。本节将深入到**操作层面**：

- **Prompt 设计方法论**：如何编写高质量的 Judge Prompt
- **打分策略对比**：直接打分 vs CoT 打分的效果差异
- **校准与验证**：如何量化 Judge 的可靠性
- **成本控制**：何时使用 LLM-as-Judge，何时使用人工

### 适用场景

- 评估集规模 > 100 条，人工逐条标注成本过高
- 需要一致性评分（同一条数据多次评估结果稳定）
- 需要频繁评估（模型迭代、Prompt 调优时反复运行）

### 已知局限性

| 偏差类型 | 说明 |
|---------|------|
| **位置偏差** | 倾向于给排在前面的选项更高的分数 |
| **冗长偏差** | 倾向于给更长的回答更高的分数 |
| **自我增强偏差** | 倾向于给与自身风格相似的回答更高的分数 |
| **格式偏差** | 结构化（如列表、表格）的回答容易获得更高分 |

### Judge Prompt 设计原则

好的 Judge Prompt 应该包含以下要素：

**1. 明确角色定义**：告诉 LLM 它是一个评估员，而不是回答者。

**2. 评分锚定**：每一档分数都有明确的文字定义，避免「看心情打分」。

```
9-10分：上下文直接且完整地回答了问题
7-8分：上下文包含主要信息
5-6分：部分相关，信息不完整
3-4分：仅有微弱关联
0-2分：完全无关
```

**3. 打分策略选择**：

- **直接打分**：Prompt 末尾直接要求返回分数。优点是速度快、token 消耗少；缺点是缺乏可解释性。
- **CoT 打分**：要求 LLM 先分析再打分。优点是评分更稳定、可解释性好；缺点是 token 消耗更多。

一般推荐使用 CoT 打分模式，特别是在校准阶段。

In [ ]:
import os
import json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from modelscope import snapshot_download
model_dir = snapshot_download('BAAI/bge-small-zh-v1.5', cache_dir='./models')
print(f"Embedding 模型路径: {model_dir}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_models import ChatZhipuAI

embedding = HuggingFaceEmbeddings(model_name=model_dir)

api_key = os.environ.get("ZHIPUAI_API_KEY")
llm = ChatZhipuAI(
    model="glm-4-flash",
    temperature=0.0,
    api_key=api_key
)
print("Embedding 和 LLM 初始化完成")

In [ ]:
import re
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

pdf_path = "../3. 索引阶段/data/pumpkin_book.pdf"
persist_dir = "./chroma_db"

def clean_text(text: str) -> str:
    text = re.sub(r'→_→\n.*?←_←', '', text, flags=re.DOTALL)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def build_vectorstore(pdf_path, embedding, persist_directory="./chroma_db"):
    """构建或加载向量库，已有则复用"""
    if os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"发现已存在的向量库: {persist_directory}，正在加载...")
        try:
            vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding)
            count = vectorstore._collection.count()
            print(f"✅ 加载成功！共 {count} 个文档块")
            return vectorstore
        except Exception as e:
            print(f"⚠️ 加载失败 ({e})，将重新构建...")
    print("开始构建向量库...")
    loader = PyMuPDFLoader(pdf_path)
    pdf_pages = loader.load()
    data_pages = pdf_pages[13:-13]
    for page in data_pages:
        page.page_content = clean_text(page.page_content)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    splits = text_splitter.split_documents(data_pages)
    vectorstore = Chroma.from_documents(documents=splits, embedding=embedding, persist_directory=persist_directory)
    print(f"✅ 向量库构建完成并保存至 {persist_directory}，共 {len(splits)} 个文档块")
    return vectorstore

vectorstore = build_vectorstore(pdf_path, embedding, persist_dir)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_template(
    "根据以下上下文回答问题。如果上下文中没有相关信息，请说'根据已有资料无法回答'。\n\n"
    "上下文：\n{context}\n\n问题：{question}\n\n回答："
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)
print("RAG Pipeline 构建完成")

In [ ]:
from tqdm import tqdm

eval_data = [
    {"question": "什么是信息增益？", "ground_truth": "信息增益是指在得知某个特征的信息后，信息不确定性减少的程度。"},
    {"question": "什么是基尼指数？", "ground_truth": "基尼指数是度量数据集纯度的一种指标，反映了从数据集中随机抽取两个样本，其类别标记不一致的概率。"},
    {"question": "过拟合和欠拟合的区别是什么？", "ground_truth": "过拟合是模型在训练集上表现好但在测试集上表现差。欠拟合是模型在训练集和测试集上都表现不好。"},
    {"question": "什么是支持向量机？", "ground_truth": "支持向量机通过在特征空间中找到一个最优超平面来实现分类，使得两类样本到超平面的间隔最大化。"},
    {"question": "朴素贝叶斯分类器的基本原理是什么？", "ground_truth": "朴素贝叶斯分类器基于贝叶斯定理，假设各特征之间条件独立，通过计算后验概率来进行分类。"},
]

results = []
for item in tqdm(eval_data, desc="生成回答"):
    docs = retriever.invoke(item["question"])
    answer = rag_chain.invoke(item["question"])
    results.append({
        "question": item["question"],
        "answer": answer,
        "contexts": [doc.page_content for doc in docs],
        "ground_truth": item["ground_truth"]
    })
print(f"已生成 {len(results)} 条回答")

## 二、手写 Context Relevance Judge

Context Relevance 评估检索到的上下文与用户问题的相关程度。我们设计两种评估方式进行对比：
- **直接打分**：直接给出 0-10 的分数
- **CoT 打分**：先分析再给分，增加可解释性

In [ ]:
def context_relevance_judge_direct(question: str, context: str) -> dict:
    """直接打分模式的 Context Relevance Judge"""
    judge_prompt = ChatPromptTemplate.from_template(
        "你是一个专业的 RAG 系统评估员。请评估以下检索到的上下文与用户问题的相关程度。\n\n"
        "评分标准（0-10 分）：\n"
        "- 9-10分：上下文直接且完整地回答了问题\n"
        "- 7-8分：上下文包含回答问题所需的主要信息\n"
        "- 5-6分：上下文部分相关，但信息不完整\n"
        "- 3-4分：上下文仅有微弱关联\n"
        "- 0-2分：上下文与问题完全无关\n\n"
        "用户问题：{question}\n\n"
        "检索到的上下文：{context}\n\n"
        "请只返回一个 0-10 的整数分数，不要返回其他内容。\n"
        "分数："
    )
    chain = judge_prompt | llm | StrOutputParser()
    score_text = chain.invoke({"question": question, "context": context})
    try:
        score = int(score_text.strip())
        score = max(0, min(10, score))
    except ValueError:
        score = 5
    return {"score": score, "mode": "direct"}

test_result = context_relevance_judge_direct(
    results[0]["question"],
    results[0]["contexts"][0]
)
print(f"直接打分测试：{test_result}")

In [ ]:
def context_relevance_judge_cot(question: str, context: str) -> dict:
    """CoT（链式思考）打分模式的 Context Relevance Judge"""
    judge_prompt = ChatPromptTemplate.from_template(
        "你是一个专业的 RAG 系统评估员。请评估以下检索到的上下文与用户问题的相关程度。\n\n"
        "评分标准（0-10 分）：\n"
        "- 9-10分：上下文直接且完整地回答了问题\n"
        "- 7-8分：上下文包含回答问题所需的主要信息\n"
        "- 5-6分：上下文部分相关，但信息不完整\n"
        "- 3-4分：上下文仅有微弱关联\n"
        "- 0-2分：上下文与问题完全无关\n\n"
        "用户问题：{question}\n\n"
        "检索到的上下文：{context}\n\n"
        "请按以下格式回答：\n"
        "分析：<分析上下文与问题的关联程度，指出哪些信息有用，哪些无关>\n"
        "分数：<0-10 的整数>"
    )
    chain = judge_prompt | llm | StrOutputParser()
    response = chain.invoke({"question": question, "context": context})

    try:
        score_line = [l for l in response.split('\n') if '分数' in l][-1]
        import re
        score = int(re.search(r'\d+', score_line).group())
        score = max(0, min(10, score))
    except (IndexError, ValueError, AttributeError):
        score = 5
    return {"score": score, "reasoning": response, "mode": "cot"}

test_result = context_relevance_judge_cot(
    results[0]["question"],
    results[0]["contexts"][0]
)
print(f"CoT 打分测试：分数={test_result['score']}")
print(f"推理过程：\n{test_result['reasoning'][:300]}")

In [ ]:
def faithfulness_judge(context: str, answer: str) -> dict:
    """Faithfulness Judge：判断答案是否基于上下文"""
    judge_prompt = ChatPromptTemplate.from_template(
        "你是一个专业的 RAG 系统评估员。请判断以下回答是否完全基于给定的上下文，还是包含了上下文中没有的信息（幻觉）。\n\n"
        "评分标准（0-10 分）：\n"
        "- 9-10分：回答完全基于上下文，每个论述都有依据\n"
        "- 7-8分：回答基本基于上下文，有少量合理推断\n"
        "- 5-6分：回答部分基于上下文，但有明显的额外信息\n"
        "- 3-4分：回答大部分内容在上下文中找不到依据\n"
        "- 0-2分：回答与上下文无关，完全是编造的\n\n"
        "上下文：{context}\n\n"
        "回答：{answer}\n\n"
        "请按以下格式回答：\n"
        "分析：<逐条检查回答中的关键论述，判断是否能在上下文中找到依据>\n"
        "分数：<0-10 的整数>"
    )
    chain = judge_prompt | llm | StrOutputParser()
    response = chain.invoke({"context": context, "answer": answer})

    try:
        score_line = [l for l in response.split('\n') if '分数' in l][-1]
        import re
        score = int(re.search(r'\d+', score_line).group())
        score = max(0, min(10, score))
    except (IndexError, ValueError, AttributeError):
        score = 5
    return {"score": score, "reasoning": response}

test_result = faithfulness_judge(
    "\n\n".join(results[0]["contexts"]),
    results[0]["answer"]
)
print(f"Faithfulness 评分：{test_result['score']}")
print(f"推理过程：\n{test_result['reasoning'][:300]}")

## 三、对比实验：直接打分 vs CoT 打分

对同一批数据分别用两种 Prompt 打分，观察分数分布差异。CoT 打分通常更稳定、更有区分度。

In [ ]:
import time
import numpy as np

direct_scores = []
cot_scores = []

for i, r in enumerate(results):
    combined_context = "\n\n".join(r["contexts"][:3])

    d = context_relevance_judge_direct(r["question"], combined_context)
    direct_scores.append(d["score"])
    time.sleep(1)

    c = context_relevance_judge_cot(r["question"], combined_context)
    cot_scores.append(c["score"])
    time.sleep(1)

    print(f"[{i+1}/{len(results)}] {r['question'][:15]}... 直接={d['score']} CoT={c['score']}")

print(f"\n直接打分 均值={sum(direct_scores)/len(direct_scores):.1f} 标准差={np.std(direct_scores):.2f}")
print(f"CoT打分  均值={sum(cot_scores)/len(cot_scores):.1f} 标准差={np.std(cot_scores):.2f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

questions_short = [q["question"][:8] + "..." for q in eval_data]
x = np.arange(len(questions_short))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, direct_scores, width, label='直接打分', color='steelblue')
bars2 = ax.bar(x + width/2, cot_scores, width, label='CoT 打分', color='coral')

ax.set_ylabel('分数')
ax.set_title('直接打分 vs CoT 打分对比')
ax.set_xticks(x)
ax.set_xticklabels(questions_short)
ax.legend()
ax.set_ylim(0, 11)

for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("./figures/judge_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
faith_scores = []
for i, r in enumerate(results):
    combined_context = "\n\n".join(r["contexts"][:3])
    result = faithfulness_judge(combined_context, r["answer"])
    faith_scores.append(result["score"])
    time.sleep(1)
    print(f"[{i+1}/{len(results)}] {r['question'][:15]}... Faithfulness={result['score']}")

print(f"\nFaithfulness 均值={sum(faith_scores)/len(faith_scores):.1f}")

## 四、校准：Cohen's Kappa 一致性验证

LLM-as-Judge 的分数可靠吗？我们通过 Cohen's Kappa 系数来衡量 Judge 评分和人工标注之间的一致性。

Cohen's Kappa 值的解读：
| Kappa 值 | 一致性水平 |
|---------|----------|
| < 0.20 | 极差 |
| 0.21-0.40 | 一般 |
| 0.41-0.60 | 中等 |
| 0.61-0.80 | 较好 |
| 0.81-1.00 | 优秀 |

In [ ]:
from sklearn.metrics import cohen_kappa_score

human_scores = [8, 7, 6, 7, 8]

def to_category(score):
    if score >= 7: return "好"
    elif score >= 4: return "中"
    else: return "差"

judge_categories = [to_category(s) for s in cot_scores]
human_categories = [to_category(s) for s in human_scores]

kappa = cohen_kappa_score(human_categories, judge_categories)

print("Judge vs 人工标注对比：")
print(f"{'问题':<20} {'Judge分数':>8} {'人工分数':>8} {'Judge类别':>8} {'人工类别':>8}")
print("-" * 60)
for i in range(len(eval_data)):
    print(f"{eval_data[i]['question'][:18]:<20} {cot_scores[i]:>8} {human_scores[i]:>8} {judge_categories[i]:>8} {human_categories[i]:>8}")
print(f"\nCohen's Kappa = {kappa:.4f}")

if kappa > 0.6:
    print("一致性较好，LLM-as-Judge 可以作为可靠的自动评估手段。")
elif kappa > 0.4:
    print("一致性中等，建议增加校准数据或优化 Judge Prompt。")
else:
    print("一致性较差，需要重新设计 Judge Prompt 或增加示例。")

## 五、小结

本节介绍了 LLM-as-Judge 的核心技术：

### 关键要点

1. **CoT 打分优于直接打分**：带推理链的评估更稳定、更有区分度
2. **评分锚定很重要**：每一档都需要明确定义，避免评分漂移
3. **需要校准**：用 Cohen's Kappa 验证 Judge 和人工的一致性
4. **成本可控**：不需要每条都评，按比例采样即可

### 何时用 LLM-as-Judge

| 场景 | 建议 |
|------|------|
| 评估集 > 100 条 | 用 LLM-as-Judge |
| 需要频繁评估（CI/CD） | 用 LLM-as-Judge |
| 高风险场景（医疗、法律） | 人工 + LLM-as-Judge 双重校验 |
| 评估标准模糊 | 先人工标注建立标准，再训练 Judge |

### 实践建议

1. **先校准再批量跑**：用 30-50 条人工标注数据校准 Judge，Kappa > 0.6 再批量使用
2. **保存推理过程**：CoT 模式的推理过程是宝贵的调试线索
3. **定期重新校准**：模型更新后，Judge 的行为可能变化

### 参考文献

- [Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685)
- [G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment](https://arxiv.org/abs/2303.16634)